# NB03 · 评测统计学：你的数字什么时候可信

| | |
|---|---|
| **目标** | 回答「多少次 rollout 的 success rate 才可信」，产出你以后一直沿用的《评测协议 v1》——评测专家的立身之本 |
| **前置** | 无硬依赖（纯 numpy，永远能跑）；有 NB02 更好 |
| **预计耗时** | 2–3 小时 |
| **产出物** | `results/NB03.json`（评测协议）+ 3 张图 |
| **通过标准** | 能脱稿回答：区分 60% 和 70% 的 policy 需要多少次 rollout |

规则：从上到下顺序执行；每个 ✅ 检查点必须核对；最后的复盘必须填写并 commit。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nbutils
rng = np.random.default_rng(0)

In [ ]:
# 实验一：同一个 policy（真实成功率 60%），不同 rollout 次数下你测到的数字有多散
p_true = 0.60
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, n in zip(axes, [20, 50, 200]):
    est = rng.binomial(n, p_true, size=2000) / n
    ax.hist(est, bins=25); ax.axvline(p_true, color="r", ls="--")
    ax.set_title(f"n={n}  std={est.std():.3f}")
plt.suptitle("同一 policy，评测 2000 次的 success rate 分布"); plt.savefig("results/NB03_spread.png", dpi=120, bbox_inches="tight"); plt.show()
# ✅ 检查点：n=20 时你可能测出 45% 也可能测出 75%——这就是为什么论文里 n=10 的对比表基本是占卜。

In [ ]:
# 实验二：置信区间宽度 vs n —— 挂在墙上的那张图
ns = np.arange(10, 501, 10)
widths = [nbutils.wilson_ci(int(0.6 * n), n)[1] - nbutils.wilson_ci(int(0.6 * n), n)[0] for n in ns]
plt.figure(figsize=(7, 3.5))
plt.plot(ns, widths); plt.axhline(0.10, color="r", ls="--", label="width=10pp")
plt.xlabel("n rollouts"); plt.ylabel("95% CI width"); plt.legend(); plt.title("要把不确定度压到 ±5pp，你需要多少 rollout？")
plt.savefig("results/NB03_ci_width.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# 实验三（功效分析）：真实差距 60% vs 70%，n 多大才能 80% 的概率"测出显著差异"
def power(p1, p2, n, trials=4000):
    a = rng.binomial(n, p1, trials) / n
    b = rng.binomial(n, p2, trials) / n
    se = np.sqrt(a*(1-a)/n + b*(1-b)/n) + 1e-9
    return float(np.mean((b - a) / se > 1.645))   # 单侧 5%

for n in [20, 50, 100, 200, 400]:
    print(f"n={n:4d}  P(检测到 60% vs 70% 的差异) = {power(0.6, 0.7, n):.0%}")
# ✅ 检查点：得出你自己的结论——"报告 A/B 对比时，每边至少 n=___"。

In [ ]:
# 实验四（若 NB02 已完成）：把理论对到你的真实 policy 上
# 用同一 checkpoint 跑 3 组独立 eval（不同 env seed），组间散布应落在 binomial 理论带内；
# 若显著超出 → 评测协议里有未控制的随机源（初始分布？渲染？非确定性推理？）——找到它。
try:
    r = nbutils.latest("NB02")
    n = r["n_eval"]; p = r["success_rate"]
    print(f"NB02: p={p:.1%}, n={n} → 理论组间 std ≈ {np.sqrt(p*(1-p)/n):.3f}")
    print("跑 3 组真实 eval，把三个数字填进下一个 cell 的 protocol 里对照。")
except FileNotFoundError as e:
    print(e)

In [ ]:
# 落盘：《我的评测协议 v1》——之后 NB04/06/07/08 全部沿用，不许临时改
PROTOCOL = {
    "n_episodes": 100,          # 根据实验二/三自己定，写清理由
    "n_seeds": 2,
    "report_format": "p% [Wilson 95% CI]",
    "ab_rule": "两组 CI 不重叠才许说'更好'；重叠只许说'无法区分'",
    "rationale": "填：为什么是这个 n？（引用实验三的数字）",
}
nbutils.log_result("NB03", PROTOCOL)
PROTOCOL

## 分析

1. 回头看 NB02 的数字：在这套协议下，它的 CI 是多少？当时若和别人的模型比，你有资格下什么结论？
2. 行业现状：很多 VLA 论文每任务 n=10–50。用实验三的表说明——它们能分辨多大的真实差距？
3. **这就是你的差异化武器**：把这套协议写进你做的每一个评测、每一篇公开报告。别人报一个数，你报一个数加一个区间，高下立判。


## 复盘（必填，不填不算完成这本 notebook）

> 复盘写在这里并 commit。允许粗糙，禁止事后美化。

- **预期 vs 实际**：
- **最大的一个意外**：
- **卡最久的一步和根因**：
- **用一句话向非技术人解释本次学到的东西**：
- **进入下一本之前要做的一个动作**：
